In [ ]:
import os
#from pathlib import Path
#import re
import pandas as pd
import numpy as np
from PIL import Image
from skimage.color import rgb2gray
from skimage.draw import disk
from matplotlib.patches import Circle
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
import math

In [ ]:
def filename(path):
    """
    Return filename without extenstion
    """
    return os.path.splitext(os.path.basename(path))[0]

def draw_foci_with_radius(image_path, df, px_size_um, output_path):
    # Load image
    image = Image.open(image_path)
    image_name = filename(image_path)
    arr = np.array(image) # convert image to numpy matrix

    # Plot image
    fig, ax = plt.subplots(figsize=(8, 8))
    ax.imshow(arr, cmap="gray")

    # Draw red circles
    for _, row in df.iterrows():

        x_px = row["x_um"] / px_size_um
        y_px = row["y_um"] / px_size_um
        r_px = row["sigma_um"] / px_size_um

        circle = Circle(
            (x_px, y_px),
            r_px,
            fill=False,
            edgecolor="red",
            linewidth=1
        )

        ax.add_patch(circle)

    # Match image coordinates
    ax.set_xlim(0, arr.shape[1])
    ax.set_ylim(arr.shape[0], 0)

    # Save imafe
    plt.savefig(
        f"{output_path}/{image_name}_sigma.png",
        dpi=300,
        bbox_inches="tight"
    )

    # Do not display image
    plt.close(fig)

def foci_one_image(image_path, df, px_size_um, output_path, plot = True):
    image = Image.open(image_path)  # load image
    image_name = filename(image_path) # get image name
    arr = np.array(image) # convert image to numpy matrix
    H, W = arr.shape # get number of pixels (512*512 for 16-bit image)

    # Storage lists
    x_list = []
    y_list = []
    sigma_list = []
    mean_list = []

    # Iteration through the thunderSTORM dataframe
    for _, row in df.iterrows():
        x_px = int(row["x_um"] / px_size_um)
        y_px = int(row["y_um"] / px_size_um)
        r_px = max(1, int(row["sigma_um"] / px_size_um)) # minimal possible value for radius is 1 pixel!

        # Build circular mask (clipped automatically)
        rr, cc = disk((y_px, x_px), r_px, shape=(H, W))
        mask = np.zeros((H, W), dtype=bool)
        mask[rr, cc] = True

        n_pixels_mask = np.sum(mask)

        # Compute mean intensity
        if n_pixels_mask > 0:
            mean_intensity = arr[mask].mean()
        else:
            mean_intensity = np.nan
        #print(x_px, x_px, r_px, n_pixels_mask, mean_intensity)

        # Add values to the corresponding lists
        x_list.append(x_px)
        y_list.append(y_px)
        sigma_list.append(r_px)
        mean_list.append(mean_intensity)
    
    # Return modified copy
    df_out = df.copy()
    df_out["x_pixel"] = x_list
    df_out["y_pixel"] = y_list
    df_out["sigma_pixel"] = sigma_list
    df_out["foci_MFI"] = mean_list

    # Make a plot and save image
    if plot:
        fig, ax = plt.subplots(figsize=(8, 8))
        ax.imshow(arr, cmap="gray")

        for _, row in df_out.iterrows():
            x = row["x_pixel"]
            y = row["y_pixel"]
            r = row["sigma_pixel"]

            circle = Circle(
            (x, y),
            r,
            fill=False,
            edgecolor="red",
            linewidth=1
            )

            ax.add_patch(circle)
        
        # Match image coordinates
        ax.set_xlim(0, arr.shape[1])
        ax.set_ylim(arr.shape[0], 0)

        # Save imafe
        plt.savefig(
            f"{output_path}/{image_name}_sigma.png",
            dpi=300,
            bbox_inches="tight"
        )

        # Do not display image
        plt.close(fig)

    return df_out

In [ ]:
image_path = "/mnt/c/users/elopatukhin/Desktop/Miscroscopy/160226_U2OS_fixed/MP_WT_0.3/C2_MP_U2OS_fixed_siORC1_WT0.3_001.nd2_(series_01)_ROI_0234-0314.tif"
px_size_um = 0.058739
output_path = "/mnt/c/users/elopatukhin/Desktop/Miscroscopy/160226_U2OS_fixed/MP_WT_0.3/foci"
image = Image.open(image_path) # open image
arr = np.array(image) # convert image to matrix
arr = np.array(image, dtype=np.float32) # convert values to float
H, W = arr.shape # get number of pixels (512*512 for 16-bit image)
px_size_um = 0.058739

In [135]:
dir_images = "/mnt/c/users/elopatukhin/Desktop/Miscroscopy/160226_U2OS_fixed/MP_WT_0.3"
dir_foci = "/mnt/c/users/elopatukhin/Desktop/Miscroscopy/160226_U2OS_fixed/MP_WT_0.3/foci"

In [144]:
paths_images = [
    os.path.join(dir_images, f)
    for f in os.listdir(dir_images)
    if os.path.isfile(os.path.join(dir_images, f))
    and f.lower().endswith(".tif") and "_ROI_".lower() in f.lower()
]

In [147]:
paths_foci_csv = [
    os.path.join(dir_foci, f)
    for f in os.listdir(dir_foci)
    if os.path.isfile(os.path.join(dir_foci, f))
    and f.lower().endswith(".csv")   
    ]

In [149]:
img_by_key = {filename(image_name): image_name for image_name in paths_images} # dictionary {image name: image path}


In [ ]:
csv_by_key = {filename(csv_name)[:-5]: csv_name for csv_name in paths_foci_csv}


In [155]:
csv_by_key

{'C2_MP_U2OS_fixed_siORC1_WT0.3_001.nd2_(series_01)_ROI_0234-0314_foci': '/mnt/c/users/elopatukhin/Desktop/Miscroscopy/160226_U2OS_fixed/MP_WT_0.3/foci/C2_MP_U2OS_fixed_siORC1_WT0.3_001.nd2_(series_01)_ROI_0234-0314_foci.csv',
 'C2_MP_U2OS_fixed_siORC1_WT0.3_001.nd2_(series_02)_ROI_0246-0281_foci': '/mnt/c/users/elopatukhin/Desktop/Miscroscopy/160226_U2OS_fixed/MP_WT_0.3/foci/C2_MP_U2OS_fixed_siORC1_WT0.3_001.nd2_(series_02)_ROI_0246-0281_foci.csv',
 'C2_MP_U2OS_fixed_siORC1_WT0.3_001.nd2_(series_03)_ROI_0248-0306_foci': '/mnt/c/users/elopatukhin/Desktop/Miscroscopy/160226_U2OS_fixed/MP_WT_0.3/foci/C2_MP_U2OS_fixed_siORC1_WT0.3_001.nd2_(series_03)_ROI_0248-0306_foci.csv',
 'C2_MP_U2OS_fixed_siORC1_WT0.3_001.nd2_(series_04)_ROI_0280-0283_foci': '/mnt/c/users/elopatukhin/Desktop/Miscroscopy/160226_U2OS_fixed/MP_WT_0.3/foci/C2_MP_U2OS_fixed_siORC1_WT0.3_001.nd2_(series_04)_ROI_0280-0283_foci.csv',
 'C2_MP_U2OS_fixed_siORC1_WT0.3_001.nd2_(series_05)_ROI_0248-0285_foci': '/mnt/c/users/elopa

In [153]:
csv_by_key

{'C2_MP_U2OS_fixed_siORC1_WT0.3_001.nd2_(series_01)_ROI_0234-0314_foci': '/mnt/c/users/elopatukhin/Desktop/Miscroscopy/160226_U2OS_fixed/MP_WT_0.3/foci/C2_MP_U2OS_fixed_siORC1_WT0.3_001.nd2_(series_01)_ROI_0234-0314_foci.csv',
 'C2_MP_U2OS_fixed_siORC1_WT0.3_001.nd2_(series_02)_ROI_0246-0281_foci': '/mnt/c/users/elopatukhin/Desktop/Miscroscopy/160226_U2OS_fixed/MP_WT_0.3/foci/C2_MP_U2OS_fixed_siORC1_WT0.3_001.nd2_(series_02)_ROI_0246-0281_foci.csv',
 'C2_MP_U2OS_fixed_siORC1_WT0.3_001.nd2_(series_03)_ROI_0248-0306_foci': '/mnt/c/users/elopatukhin/Desktop/Miscroscopy/160226_U2OS_fixed/MP_WT_0.3/foci/C2_MP_U2OS_fixed_siORC1_WT0.3_001.nd2_(series_03)_ROI_0248-0306_foci.csv',
 'C2_MP_U2OS_fixed_siORC1_WT0.3_001.nd2_(series_04)_ROI_0280-0283_foci': '/mnt/c/users/elopatukhin/Desktop/Miscroscopy/160226_U2OS_fixed/MP_WT_0.3/foci/C2_MP_U2OS_fixed_siORC1_WT0.3_001.nd2_(series_04)_ROI_0280-0283_foci.csv',
 'C2_MP_U2OS_fixed_siORC1_WT0.3_001.nd2_(series_05)_ROI_0248-0285_foci': '/mnt/c/users/elopa